# Latih Semua Model Web Deteksi Perilaku

Notebook ini melatih **kedua model** yang dipakai web:

| Model | Kelas | Mode web | Berkas di web |
| --- | --- | --- | --- |
| A. Parkiran | person, motorcycle | parkiran motor | `web/models/motor.pt` |
| B. Umum | person | koridor, toko, rumah | `web/models/best.pt` |

## Cara menjalankan

1. **Runtime > Change runtime type > GPU** (T4 atau L4).
2. **Runtime > Run all**.
3. **Bila Colab terputus, runtime di-reset, atau sesi habis:** sambungkan lagi, lalu **Runtime > Run all** lagi. Tidak ada yang diulang dari awal:

| Tahap | Disimpan di Drive | Saat dijalankan ulang |
| --- | --- | --- |
| Unduhan COCO dan Open Images | `cache_data/*.tar` per sumber | Diambil dari Drive, tidak diunduh lagi |
| Dataset jadi (dengan oklusi, malam, CCTV) | `cache_data/data_final.tar` | Diekstrak dalam beberapa menit |
| Pelatihan | `detect/motor-3`, `detect/person-3` (checkpoint tiap epoch) | Lanjut dari epoch terakhir |
| Model yang sudah selesai | penanda `SELESAI` | Dilewati |

## Perbedaan dengan notebook sebelumnya

| Masalah | Perbaikan |
| --- | --- |
| Runtime mati saat pelatihan dimulai, semua data dan variabel hilang | Pelatihan berjalan di proses terpisah; kernel notebook tetap hidup |
| Unduhan 1 sampai 2 jam hilang setiap runtime di-reset | Setiap sumber disimpan ke Drive setelah selesai diunduh |
| `NameError` bila sel dijalankan tidak berurutan | Tiap sel membaca pengaturan dari berkas, bukan dari variabel sel lain |
| Harus memilih sel latih atau lanjutkan | Satu sel per model: mulai, lanjut, atau lewati otomatis |
| `!python` gagal tetapi Run all jalan terus | Sel berhenti dengan error bila skrip gagal |

## Yang harus ada di Google Drive

```
MyDrive/supicious-activity-detection/
  dataset_yolo_person_v2/            dataset orang yang sudah diperbaiki
  detect/motor-1/weights/best.pt     model parkiran lama (pembanding)
  detect/train-4/weights/best.pt     model orang lama (pembanding)
  video_uji/                         minimal 1 video uji
```

Sediakan ruang kosong Drive minimal **5 GB** untuk cache data. Gambar disimpan dengan sisi terpanjang 640 piksel (ukuran latih), jadi tidak ada mutu yang hilang.

## Sumber data

| Sumber | Isi | Lisensi |
| --- | --- | --- |
| `dataset_yolo_person_v2` | Orang di koridor dan lobi (CAVIAR) | CC BY-SA, EC Funded CAVIAR project/IST 2001 37540 |
| COCO 2017 | Motor, orang, sepeda (negatif), benda penghalang | Anotasi CC BY 4.0; gambar lisensi Flickr |
| Open Images V7 | Orang tertutup sebagian (IsOccluded), motor | Anotasi CC BY 4.0; gambar CC BY 2.0 |

## 1. Cek GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU tidak aktif. Pilih Runtime > Change runtime type > GPU, lalu Run all lagi.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Google Drive, pengaturan, dan pemeriksaan bahan

Semua pengaturan ditulis ke `/content/sad_config.json` dan dibaca oleh setiap sel serta skrip. Nilai sudah ditetapkan dan tidak perlu diubah.

In [ ]:
import json, os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/supicious-activity-detection')
KONFIG = {
    # lokasi
    'drive_dir': DRIVE_DIR,
    'cache_dir': DRIVE_DIR / 'cache_data',          # cache unduhan dan dataset jadi
    'hasil_dir': DRIVE_DIR / 'detect',              # hasil pelatihan
    'paket_dir': DRIVE_DIR / 'model_web',           # model siap pasang
    'data_person': DRIVE_DIR / 'dataset_yolo_person_v2',
    'video_dir': DRIVE_DIR / 'video_uji',
    'model_lama': {'A': DRIVE_DIR / 'detect/motor-1/weights/best.pt', 'B': DRIVE_DIR / 'detect/train-4/weights/best.pt'},
    'kerja_dir': Path('/content'),
    'data_dir': Path('/content/data_v3'),           # dataset model A, disk Colab
    'data_b': Path('/content/data_v3_person'),      # dataset model B, memakai gambar yang sama
    # data
    'kelas': ['person', 'motorcycle'],
    'coco_latih': {'motorcycle': 3000, 'bicycle': 500, 'person': 3000},   # sepeda = contoh negatif
    'coco_val': {'motorcycle': 200, 'person': 400},
    'oi_latih': {'Motorcycle': 1500, 'Person': 3000},   # Person = orang berlabel IsOccluded
    'oi_val': {'Motorcycle': 300, 'Person': 500},
    'penghalang': {'motorcycle': 300, 'car': 300, 'chair': 200, 'bench': 200},
    'oklusi': 0.30, 'simulasi_malam': 0.25, 'simulasi_cctv': 0.20,
    'sisi_simpan': 640,
    'min': {'orang_latih': 10000, 'motor_latih': 3000, 'motor_val': 300, 'oklusi_val': 100, 'caviar_val': 90,
            'bank': 200, 'oklusi_latih': 500},
    # pelatihan
    'model_awal': 'yolo26s.pt',
    'nama': {'A': 'motor-3', 'B': 'person-3'},
    'berkas_web': {'A': 'motor.pt', 'B': 'best.pt'},
    'epochs': 80, 'imgsz': 640, 'batch': 16, 'patience': 20, 'seed': 51,
    'workers': min(4, os.cpu_count() or 2),
}

kurang = []
if not (KONFIG['data_person'] / 'data.yaml').exists():
    kurang.append(f"{KONFIG['data_person']} (unggah folder dataset_yolo_person_v2)")
for m, p in KONFIG['model_lama'].items():
    if not p.exists():
        kurang.append(f'{p} (model lama untuk pembanding model {m})')
video = [p for p in KONFIG['video_dir'].glob('*') if p.suffix.lower() in ('.mp4', '.avi', '.mov', '.mkv', '.mpg')]
if not video:
    kurang.append(f"{KONFIG['video_dir']} berisi minimal 1 video uji")
assert not kurang, 'Lengkapi dulu di Google Drive:\n- ' + '\n- '.join(kurang)
for kunci in ('cache_dir', 'hasil_dir', 'paket_dir'):
    KONFIG[kunci].mkdir(parents=True, exist_ok=True)

def ke_teks(v):
    return {k: ke_teks(x) for k, x in v.items()} if isinstance(v, dict) else (str(v) if isinstance(v, Path) else v)
Path('/content/sad_config.json').write_text(json.dumps(ke_teks(KONFIG), indent=1))

final = KONFIG['cache_dir'] / 'data_final.tar'
print('Bahan lengkap. Video uji:', sorted(v.name for v in video))
print('Cache dataset jadi di Drive:', 'ADA, data tidak diunduh lagi' if final.with_name(final.name + '.ok').exists() else 'belum ada')

## 3. Instal pustaka

Ultralytics dikunci di 8.4.194, sama dengan web. FiftyOne (pengunduh COCO dan Open Images) hanya dipasang bila dataset jadi belum ada di Drive.

In [ ]:
import json
from pathlib import Path
k = json.loads(Path('/content/sad_config.json').read_text())
final = Path(k['cache_dir']) / 'data_final.tar'
PAKET_PIP = 'ultralytics==8.4.194' + ('' if final.with_name(final.name + '.ok').exists() else ' fiftyone')
!pip install -q {PAKET_PIP}
import ultralytics
ultralytics.checks()

## 4. Fungsi bersama

Ditulis ke `/content/sad_bantu.py` dan dipakai oleh semua sel dan skrip.

In [ ]:
%%writefile /content/sad_bantu.py
"""Fungsi bersama notebook, skrip data, skrip latih, dan skrip evaluasi."""
import json, os, shutil, tarfile, time
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import yaml

KONFIG = Path('/content/sad_config.json')
EKSTENSI = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
DATA_VERSI = 'v3-hardlink'   # penanda versi data pada folder hasil pelatihan


def muat_konfig():
    assert KONFIG.exists(), 'Konfigurasi belum ada. Jalankan sel 3 (Pengaturan) dulu.'
    k = json.loads(KONFIG.read_text())
    for kunci in ('drive_dir', 'cache_dir', 'hasil_dir', 'paket_dir', 'data_dir', 'data_b', 'data_person',
                  'video_dir', 'kerja_dir'):
        k[kunci] = Path(k[kunci])
    k['model_lama'] = {m: Path(p) for m, p in k['model_lama'].items()}
    return k


def jalankan(*perintah):
    """Jalankan skrip Python di proses terpisah dan tampilkan keluarannya langsung di sel.
    Bila proses gagal, sel berhenti dengan error sehingga 'Run all' tidak melanjutkan ke sel berikutnya."""
    import codecs, subprocess, sys
    p = subprocess.Popen([sys.executable, '-u', *map(str, perintah)], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    dekoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        while True:
            potongan = p.stdout.read1(65536)
            if not potongan:
                break
            sys.stdout.write(dekoder.decode(potongan))
            sys.stdout.flush()
        kode = p.wait()
    except BaseException:
        p.terminate()
        p.wait()
        raise
    if kode != 0:
        raise RuntimeError(f'{Path(str(perintah[0])).name} berhenti dengan kode {kode}. Baca pesan di atas.')


# ---------- cache di Google Drive ----------
# Setiap berkas cache punya penanda <nama>.ok berisi ukuran berkas. Berkas tanpa penanda,
# atau yang ukurannya tidak cocok (salinan terputus), dianggap tidak ada dan dibuat ulang.

def _penanda(tujuan):
    return tujuan.with_name(tujuan.name + '.ok')


def cache_ada(tujuan):
    tujuan = Path(tujuan)
    p = _penanda(tujuan)
    if not (tujuan.exists() and p.exists()):
        return False
    try:
        return json.loads(p.read_text())['ukuran'] == tujuan.stat().st_size
    except Exception:
        return False


def simpan_cache(sumber, tujuan):
    """Salin berkas lokal ke Drive, lalu tulis penanda. Penanda lama dihapus lebih dulu."""
    sumber, tujuan = Path(sumber), Path(tujuan)
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    _penanda(tujuan).unlink(missing_ok=True)
    sementara = tujuan.with_name(tujuan.name + '.sementara')
    shutil.copyfile(sumber, sementara)
    try:
        os.replace(sementara, tujuan)
    except OSError:   # bila Drive menolak ganti nama, salin langsung; penanda tetap ditulis paling akhir
        shutil.copyfile(sumber, tujuan)
        sementara.unlink(missing_ok=True)
    _penanda(tujuan).write_text(json.dumps({'ukuran': sumber.stat().st_size, 'waktu': time.ctime()}))
    assert cache_ada(tujuan), f'Gagal menyimpan cache {tujuan}'


def kemas(folder, tujuan_drive, kerja_dir):
    """Kemas folder menjadi tar (tanpa kompresi, gambar sudah JPEG) lalu simpan ke Drive."""
    folder = Path(folder)
    lokal = Path(kerja_dir) / 'kemas' / Path(tujuan_drive).name
    lokal.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(lokal, 'w') as t:
        for f in sorted(folder.rglob('*')):
            if f.is_file() and not f.name.endswith('.cache'):
                t.add(f, arcname=str(f.relative_to(folder)))
    simpan_cache(lokal, tujuan_drive)
    ukuran = lokal.stat().st_size / 1e6
    lokal.unlink()
    print(f'  disimpan ke Drive: {Path(tujuan_drive).name} ({ukuran:.0f} MB)')


def buka(tar_drive, folder, kerja_dir):
    """Salin tar dari Drive ke disk lokal (lebih cepat dari membaca langsung), lalu ekstrak."""
    assert cache_ada(tar_drive), f'Cache {tar_drive} tidak lengkap'
    lokal = Path(kerja_dir) / 'kemas' / Path(tar_drive).name
    lokal.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tar_drive, lokal)
    Path(folder).mkdir(parents=True, exist_ok=True)
    with tarfile.open(lokal) as t:
        t.extractall(folder, filter='data')
    lokal.unlink()


def unduh_cache(url, tujuan_drive, kerja_dir):
    """Unduh sekali ke Drive. Pada jalankan ulang, berkas diambil dari Drive."""
    import urllib.request
    tujuan_drive = Path(tujuan_drive)
    if not cache_ada(tujuan_drive):
        lokal = Path(kerja_dir) / 'unduhan' / tujuan_drive.name
        lokal.parent.mkdir(parents=True, exist_ok=True)
        print('Mengunduh', url)
        for coba in range(1, 6):
            try:
                urllib.request.urlretrieve(url, lokal)
                break
            except Exception as e:
                assert coba < 5, f'Gagal mengunduh {url}: {e}'
                print(f'  gagal ({e}), coba lagi dalam {10 * coba} detik')
                time.sleep(10 * coba)
        simpan_cache(lokal, tujuan_drive)
        lokal.unlink()
    return tujuan_drive


# ---------- format YOLO ----------

def tulis_gambar(sumber, tujuan, sisi_maks):
    """Tulis gambar sebagai JPEG. Sisi terpanjang diperkecil ke sisi_maks (ukuran latih) agar cache kecil."""
    img = cv2.imread(str(sumber))
    if img is None:
        return False
    h, w = img.shape[:2]
    if max(h, w) > sisi_maks:
        f = sisi_maks / max(h, w)
        img = cv2.resize(img, (max(1, round(w * f)), max(1, round(h * f))), interpolation=cv2.INTER_AREA)
    return cv2.imwrite(str(tujuan), img, [cv2.IMWRITE_JPEG_QUALITY, 92])


def tulis_contoh(akar, split, nama, sumber_gambar, baris, sisi_maks):
    (Path(akar) / 'images' / split).mkdir(parents=True, exist_ok=True)
    (Path(akar) / 'labels' / split).mkdir(parents=True, exist_ok=True)
    if not tulis_gambar(sumber_gambar, Path(akar) / 'images' / split / f'{nama}.jpg', sisi_maks):
        return False
    (Path(akar) / 'labels' / split / f'{nama}.txt').write_text(''.join(b + '\n' for b in baris))
    return True


def baca_label(p):
    out = []
    p = Path(p)
    if p.exists():
        for b in p.read_text().splitlines():
            q = b.split()
            if len(q) >= 5:
                out.append([int(float(q[0]))] + [float(v) for v in q[1:5]])
    return out


def hitung(akar, split, kelas):
    jumlah, gambar = Counter(), 0
    for f in (Path(akar) / 'labels' / split).glob('*.txt'):
        gambar += 1
        for b in f.read_text().splitlines():
            if b.strip():
                jumlah[kelas[int(b.split()[0])]] += 1
    return gambar, dict(jumlah)


def tulis_yaml(akar, val, kelas, nama_berkas):
    y = Path(akar) / nama_berkas
    with open(y, 'w') as f:
        yaml.safe_dump({'path': str(akar), 'train': 'images/train', 'val': f'images/{val}',
                        'names': {i: n for i, n in enumerate(kelas)}}, f, sort_keys=False)
    return y


def versi_person(src, dst):
    """Dataset khusus kelas person: folder gambar ditautkan, label motor dibuang."""
    src, dst = Path(src), Path(dst)
    if dst.exists():
        shutil.rmtree(dst)
    (dst / 'images').mkdir(parents=True)
    for split_dir in sorted((src / 'images').iterdir()):
        (dst / 'images' / split_dir.name).mkdir()
        for f in split_dir.iterdir():
            try:
                os.link(f, dst / 'images' / split_dir.name / f.name)
            except OSError:
                shutil.copy2(f, dst / 'images' / split_dir.name / f.name)
        out = dst / 'labels' / split_dir.name
        out.mkdir(parents=True)
        for f in (src / 'labels' / split_dir.name).glob('*.txt'):
            out.joinpath(f.name).write_text(''.join(b + '\n' for b in f.read_text().splitlines()
                                                    if b.strip() and b.split()[0] == '0'))


def hapus_cache_label(akar):
    """Hapus labels/*.cache buatan Ultralytics. Cache lama bisa menyimpan hasil pemeriksaan yang salah."""
    for c in (Path(akar) / 'labels').glob('*.cache'):
        c.unlink()


def periksa_dataset(data_yaml):
    """Pastikan folder gambar bukan tautan simbolik dan semua label berada dalam jumlah kelas."""
    with open(data_yaml) as f:
        d = yaml.safe_load(f)
    akar, nc = Path(d['path']), len(d['names'])
    for kunci in ('train', 'val'):
        img = akar / d[kunci]
        assert img.is_dir(), f'Folder {img} tidak ada'
        assert not img.is_symlink(), f'{img} masih tautan simbolik. Jalankan ulang dari sel 1 dengan notebook terbaru.'
        lbl = akar / 'labels' / img.name
        salah = [f.name for f in lbl.glob('*.txt')
                 if any(b.strip() and int(b.split()[0]) >= nc for b in f.read_text().splitlines())]
        assert not salah, f'{len(salah)} label di {lbl} memakai kelas di luar 0-{nc - 1}, contoh {salah[:3]}'
    print(f'Data {Path(data_yaml).name} di {akar.name}: struktur benar, semua label kelas 0-{nc - 1}.')


def siapkan_yaml(k):
    """data.yaml untuk model A (person + motorcycle) dan model B (person) beserta set uji khususnya."""
    hapus_cache_label(k['data_dir'])
    versi_person(k['data_dir'], k['data_b'])
    hasil = {}
    for m, akar, kelas in (('A', k['data_dir'], k['kelas']), ('B', k['data_b'], ['person'])):
        hasil[m] = {'validasi': tulis_yaml(akar, 'val', kelas, 'data.yaml'),
                    'val_oklusi': tulis_yaml(akar, 'val_oklusi', kelas, 'data_oklusi.yaml'),
                    'val_caviar': tulis_yaml(akar, 'val_caviar', kelas, 'data_caviar.yaml')}
    return hasil


# ---------- simulasi kondisi CCTV ----------

def jadikan_malam(img, rng):
    """Inframerah malam: hitam-putih, gelap, berderau."""
    abu = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    abu = abu ** rng.uniform(1.3, 2.2)
    abu = abu * rng.uniform(0.6, 1.0) + rng.uniform(0.0, 0.08)
    abu = abu + rng.normal(0, rng.uniform(0.02, 0.06), abu.shape)
    abu = np.clip(abu * 255, 0, 255).astype(np.uint8)
    if rng.random() < 0.5:
        abu = cv2.GaussianBlur(abu, (3, 3), 0)
    return cv2.cvtColor(abu, cv2.COLOR_GRAY2BGR)


def jadikan_cctv(img, rng):
    """CCTV murah: resolusi rendah, kompresi DVR, sedikit buram."""
    h, w = img.shape[:2]
    f = rng.uniform(0.35, 0.6)
    kecil = cv2.resize(img, (max(16, int(w * f)), max(16, int(h * f))), interpolation=cv2.INTER_AREA)
    img = cv2.resize(kecil, (w, h), interpolation=cv2.INTER_LINEAR)
    img = np.clip(img.astype(np.float32) * rng.uniform(0.85, 1.1, 3) + rng.uniform(-15, 15), 0, 255).astype(np.uint8)
    if rng.random() < 0.5:
        img = cv2.GaussianBlur(img, (3, 3), 0)
    ok, buf = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(20, 55))])
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)


# ---------- status pelatihan ----------

def status_latih(k, m):
    """'belum', 'berjalan' (bisa dilanjutkan), atau 'selesai'."""
    import torch
    run = k['hasil_dir'] / k['nama'][m]
    last, best = run / 'weights' / 'last.pt', run / 'weights' / 'best.pt'
    if (run / 'SELESAI').exists() and best.exists():
        return 'selesai'
    if not last.exists():
        return 'belum'
    try:
        ck = torch.load(str(last), map_location='cpu', weights_only=False)
    except Exception:
        return 'rusak'
    if ck.get('optimizer') is None or ck.get('epoch', -1) == -1:
        return 'selesai' if best.exists() else 'rusak'
    return 'berjalan'


def model_terbaik(k, m):
    p = k['hasil_dir'] / k['nama'][m] / 'weights' / 'best.pt'
    assert status_latih(k, m) == 'selesai' and p.exists(), \
        f'Model {m} belum selesai dilatih. Jalankan sel latih model {m} sampai selesai.'
    return p

## 5. Skrip penyiapan data

Urutan kerja skrip:

1. Bila `data_final.tar` ada di Drive, ekstrak saja.
2. Bila belum, ambil setiap sumber dari cache Drive. Sumber yang belum ada diunduh lalu langsung disimpan ke Drive.
3. Rakit dataset: COCO, Open Images, CAVIAR, oklusi sintetis 30% (Ghiasi dkk., CVPR 2021), simulasi malam 25%, simulasi CCTV 20%.
4. Simpan dataset jadi ke Drive.

ID gambar dipilih dengan seed tetap, jadi hasilnya sama setiap kali dijalankan. Gambar dengan kotak kelompok (IsGroupOf) atau gambar/patung (IsDepiction) dibuang. Kotak kerumunan COCO (iscrowd) dibuang seperti konversi resmi Ultralytics.

In [ ]:
%%writefile /content/siapkan_data.py
"""Penyiapan data latih. Setiap tahap disimpan ke Google Drive, jadi jalankan ulang tidak mengunduh dari awal.

Urutan pemulihan:
1. data_final.tar ada di Drive  -> ekstrak saja (beberapa menit).
2. Cache sumber ada di Drive     -> rakit ulang tanpa mengunduh.
3. Tidak ada cache               -> unduh sumber yang belum ada, simpan ke Drive, lalu rakit.
"""
import json, random, shutil, sys, zipfile
from collections import Counter
from pathlib import Path

import cv2
import numpy as np

sys.path.insert(0, '/content')
from sad_bantu import (EKSTENSI, baca_label, buka, cache_ada, hitung, jadikan_cctv, jadikan_malam, kemas,
                       muat_konfig, siapkan_yaml, tulis_contoh, unduh_cache)

K = muat_konfig()
KELAS, SEED, SISI = K['kelas'], K['seed'], K['sisi_simpan']
DATA, CACHE, KERJA = K['data_dir'], K['cache_dir'], K['kerja_dir']
TAHAP = KERJA / 'tahap'          # folder sementara tiap sumber sebelum dikemas
CACHE.mkdir(parents=True, exist_ok=True)

FINAL = CACHE / 'data_final.tar'
SUMBER = {n: CACHE / f'{n}.tar' for n in ('coco_train', 'coco_val', 'oi_train', 'oi_val', 'bank')}


def selesai_ringkas():
    yamls = siapkan_yaml(K)
    (KERJA / 'sad_yaml.json').write_text(json.dumps({m: {d: str(p) for d, p in v.items()} for m, v in yamls.items()}))
    print('\nRingkasan data:')
    for split in ('train', 'val', 'val_oklusi', 'val_caviar'):
        g, j = hitung(DATA, split, KELAS)
        print(f'  {split:11s}: {g:6d} gambar | {j}')
    gl, kl = hitung(DATA, 'train', KELAS)
    gv, kv = hitung(DATA, 'val', KELAS)
    go, _ = hitung(DATA, 'val_oklusi', KELAS)
    gc, _ = hitung(DATA, 'val_caviar', KELAS)
    assert kl.get('person', 0) >= K['min']['orang_latih'], 'Orang di data latih terlalu sedikit'
    assert kl.get('motorcycle', 0) >= K['min']['motor_latih'], 'Motor di data latih terlalu sedikit'
    assert kv.get('motorcycle', 0) >= K['min']['motor_val'], 'Motor di validasi terlalu sedikit'
    assert go >= K['min']['oklusi_val'], 'val_oklusi terlalu sedikit'
    assert gc >= K['min']['caviar_val'], 'Validasi CAVIAR tidak lengkap'
    print('DATA SIAP')


# ---------- 1. pulihkan dari cache akhir ----------
if cache_ada(FINAL):
    print('Memulihkan data dari Drive (data_final.tar), tanpa mengunduh ...')
    if DATA.exists():
        shutil.rmtree(DATA)
    buka(FINAL, DATA, KERJA)
    selesai_ringkas()
    sys.exit(0)


# ---------- FiftyOne hanya dimuat bila ada sumber yang belum di-cache ----------
FO = None


def fiftyone():
    global FO
    if FO is None:
        import fiftyone as fo
        import fiftyone.zoo as foz
        FO = (fo, foz)
    return FO


def muat(nama_zoo, split, ids, nama):
    fo, foz = fiftyone()
    if fo.dataset_exists(nama):
        fo.delete_dataset(nama)
    try:
        return foz.load_zoo_dataset(nama_zoo, split=split, label_types=['detections'], image_ids=list(ids),
                                    dataset_name=nama, num_workers=16)
    except TypeError:   # versi FiftyOne tanpa num_workers
        return foz.load_zoo_dataset(nama_zoo, split=split, label_types=['detections'], image_ids=list(ids),
                                    dataset_name=nama)


def field_label(ds):
    skema = ds.get_field_schema()
    for f in ('ground_truth', 'detections'):
        if f in skema:
            return f
    raise RuntimeError(f'Field label tidak ditemukan: {list(skema)}')


def atribut(det, nama):
    return bool(getattr(det, nama, False))


def ekspor(ds, ke_kelas, akar, split, awalan, split_oklusi=None):
    """Tulis sampel FiftyOne ke format YOLO. Gambar dengan kotak kelompok (IsGroupOf) atau gambar/patung
    (IsDepiction) dibuang; kotak kerumunan COCO (iscrowd) dibuang seperti konversi COCO resmi Ultralytics."""
    field = field_label(ds)
    n = n_okl = 0
    for s in ds:
        dets = s[field].detections if s[field] is not None else []
        dipakai = [d for d in dets if d.label in ke_kelas and not getattr(d, 'iscrowd', 0)]
        if any(atribut(d, 'IsGroupOf') or atribut(d, 'IsDepiction') for d in dipakai):
            continue
        baris = []
        for d in dipakai:
            x, y, w, h = d.bounding_box   # kiri-atas, lebar, tinggi (0-1)
            if w > 0 and h > 0:
                baris.append(f'{KELAS.index(ke_kelas[d.label])} {x + w / 2:.6f} {y + h / 2:.6f} {w:.6f} {h:.6f}')
        nama = f'{awalan}_{Path(s.filepath).stem}'
        if tulis_contoh(akar, split, nama, s.filepath, baris, SISI):
            n += 1
            tertutup = any(atribut(d, 'IsOccluded') for d in dipakai if ke_kelas[d.label] == 'person')
            if split_oklusi and tertutup:
                tulis_contoh(akar, split_oklusi, nama, s.filepath, baris, SISI)
                n_okl += 1
    return n, n_okl


# ---------- 2. COCO 2017 ----------
COCO_RAW = Path('/root/fiftyone/coco-2017/raw')   # FiftyOne membaca anotasi dari sini


def anotasi_coco():
    if (COCO_RAW / 'instances_train2017.json').exists() and (COCO_RAW / 'instances_val2017.json').exists():
        return
    zp = unduh_cache('http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
                     CACHE / 'unduhan' / 'annotations_trainval2017.zip', KERJA)
    COCO_RAW.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        for nama in ('annotations/instances_train2017.json', 'annotations/instances_val2017.json'):
            with z.open(nama) as src, open(COCO_RAW / Path(nama).name, 'wb') as dst:
                shutil.copyfileobj(src, dst)


def pilih_coco(berkas, *kelompok):
    """Pilih ID gambar per kelas dengan seed tetap. Dalam satu kelompok, satu gambar tidak dipilih dua kali."""
    d = json.load(open(COCO_RAW / berkas))
    nama_kat = {c['id']: c['name'] for c in d['categories']}
    isi = {}
    for a in d['annotations']:
        if not a.get('iscrowd', 0):
            isi.setdefault(a['image_id'], set()).add(nama_kat[a['category_id']])
    del d
    urutan = sorted(isi)
    random.Random(SEED).shuffle(urutan)
    hasil = {}
    for permintaan in kelompok:
        dipakai = set()
        for kunci, kelas, jumlah in permintaan:
            ids = [i for i in urutan if kelas in isi[i] and i not in dipakai][:jumlah]
            dipakai.update(ids)
            hasil[kunci] = ids
    for kunci, ids in hasil.items():
        assert ids, f'Tidak ada gambar COCO untuk {kunci}'
    return hasil


COCO_KE_KELAS = {'person': 'person', 'motorcycle': 'motorcycle'}
perlu_coco = [n for n in ('coco_train', 'coco_val', 'bank') if not cache_ada(SUMBER[n])]
if perlu_coco:
    anotasi_coco()
    pilih_tr = pilih_coco('instances_train2017.json', [(c, c, n) for c, n in K['coco_latih'].items()],
                          [(f'bank_{c}', c, n) for c, n in K['penghalang'].items()])
    pilih_va = pilih_coco('instances_val2017.json', [(c, c, n) for c, n in K['coco_val'].items()])

    for nama, split_zoo, split, ids in (
            ('coco_train', 'train', 'train', [i for c in K['coco_latih'] for i in pilih_tr[c]]),
            ('coco_val', 'validation', 'val', [i for c in K['coco_val'] for i in pilih_va[c]])):
        if cache_ada(SUMBER[nama]):
            continue
        print(f'\n[{nama}] mengunduh {len(ids)} gambar COCO')
        akar = TAHAP / nama
        shutil.rmtree(akar, ignore_errors=True)
        ds = muat('coco-2017', split_zoo, ids, nama)
        n, _ = ekspor(ds, COCO_KE_KELAS, akar, split, 'coco')
        print(f'  {len(ds)} diunduh, {n} ditulis')
        assert n >= 0.95 * len(ids), f'Sebagian besar gambar {nama} gagal diunduh. Jalankan sel ini lagi.'
        kemas(akar, SUMBER[nama], KERJA)
        shutil.rmtree(akar)

    if not cache_ada(SUMBER['bank']):
        # potongan benda penghalang (motor, mobil, kursi, bangku) untuk oklusi sintetis
        print('\n[bank] mengunduh gambar benda penghalang COCO')
        ids = sorted({i for c in K['penghalang'] for i in pilih_tr[f'bank_{c}']})
        ds = muat('coco-2017', 'train', ids, 'coco_bank')
        f = field_label(ds)
        akar = TAHAP / 'bank'
        shutil.rmtree(akar, ignore_errors=True)
        akar.mkdir(parents=True)
        jumlah = Counter()
        for s in ds:
            img = cv2.imread(s.filepath)
            if img is None or s[f] is None:
                continue
            H, W = img.shape[:2]
            for d in s[f].detections:
                if d.label not in K['penghalang'] or getattr(d, 'iscrowd', 0):
                    continue
                x, y, w, h = d.bounding_box
                x1, y1, x2, y2 = int(x * W), int(y * H), int((x + w) * W), int((y + h) * H)
                if x2 - x1 >= 48 and y2 - y1 >= 48:
                    cv2.imwrite(str(akar / f'{d.label}__{jumlah[d.label]:05d}.jpg'), img[y1:y2, x1:x2],
                                [cv2.IMWRITE_JPEG_QUALITY, 95])
                    jumlah[d.label] += 1
        print('  potongan:', dict(jumlah))
        assert sum(jumlah.values()) >= K['min']['bank'], 'Bank penghalang terlalu kecil'
        kemas(akar, SUMBER['bank'], KERJA)
        shutil.rmtree(akar)


# ---------- 3. Open Images V7: motor dan orang tertutup sebagian ----------
OI_KE_KELAS = {'Person': 'person', 'Man': 'person', 'Woman': 'person', 'Boy': 'person', 'Girl': 'person',
               'Motorcycle': 'motorcycle'}
OI_ANOTASI = {'test': 'https://storage.googleapis.com/openimages/v5/test-annotations-bbox.csv',
              'validation': 'https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv'}

if not (cache_ada(SUMBER['oi_train']) and cache_ada(SUMBER['oi_val'])):
    import pandas as pd
    kelas_oi = pd.read_csv(unduh_cache('https://storage.googleapis.com/openimages/v5/class-descriptions-boxable.csv',
                                       CACHE / 'unduhan' / 'oi_kelas.csv', KERJA), header=None, names=['mid', 'nama'])
    MID_KE_NAMA = dict(zip(kelas_oi.mid, kelas_oi.nama))
    assert all(n in MID_KE_NAMA.values() for n in OI_KE_KELAS), 'Nama kelas Open Images tidak ditemukan'

    def pilih_oi(split, permintaan):
        df = pd.read_csv(unduh_cache(OI_ANOTASI[split], CACHE / 'unduhan' / f'oi_{split}_bbox.csv', KERJA),
                         usecols=['ImageID', 'LabelName', 'IsOccluded', 'IsGroupOf', 'IsDepiction'])
        df['nama'] = df.LabelName.map(MID_KE_NAMA)
        df = df[df.nama.isin(OI_KE_KELAS)]
        buruk = set(df[(df.IsGroupOf == 1) | (df.IsDepiction == 1)].ImageID)
        df = df[~df.ImageID.isin(buruk)]
        kelompok = {'Motorcycle': set(df[df.nama == 'Motorcycle'].ImageID),
                    'Person': set(df[(df.nama.map(OI_KE_KELAS) == 'person') & (df.IsOccluded == 1)].ImageID)}
        dipakai, hasil = set(), {}
        for kelas, jumlah in permintaan.items():
            ids = sorted(kelompok[kelas] - dipakai)
            random.Random(SEED).shuffle(ids)
            hasil[kelas] = ids[:jumlah]
            dipakai.update(hasil[kelas])
            assert hasil[kelas], f'Pilihan Open Images {split} {kelas} kosong'
        return hasil

    for nama, split_oi, split, permintaan in (('oi_train', 'test', 'train', K['oi_latih']),
                                              ('oi_val', 'validation', 'val', K['oi_val'])):
        if cache_ada(SUMBER[nama]):
            continue
        pilih = pilih_oi(split_oi, permintaan)
        ids = [i for c in permintaan for i in pilih[c]]
        print(f'\n[{nama}] mengunduh {len(ids)} gambar Open Images ({split_oi})')
        akar = TAHAP / nama
        shutil.rmtree(akar, ignore_errors=True)
        ds = muat('open-images-v7', split_oi, ids, nama)
        n, n_okl = ekspor(ds, OI_KE_KELAS, akar, split, 'oi', split_oklusi='val_oklusi' if split == 'val' else None)
        print(f'  {len(ds)} diunduh, {n} ditulis' + (f', {n_okl} masuk val_oklusi' if split == 'val' else ''))
        assert n >= 0.9 * len(ids), f'Sebagian besar gambar {nama} gagal diunduh. Jalankan sel ini lagi.'
        kemas(akar, SUMBER[nama], KERJA)
        shutil.rmtree(akar)


# ---------- 4. rakit dataset ----------
print('\nMerakit dataset dari cache Drive ...')
if DATA.exists():
    shutil.rmtree(DATA)
for s in ('train', 'val', 'val_oklusi', 'val_caviar'):
    (DATA / 'images' / s).mkdir(parents=True)
    (DATA / 'labels' / s).mkdir(parents=True)
for nama in ('coco_train', 'coco_val', 'oi_train', 'oi_val'):
    buka(SUMBER[nama], DATA, KERJA)

# dataset_yolo_person_v2 (CAVIAR) dari Drive; split test ikut validasi
import yaml
with open(K['data_person'] / 'data.yaml') as f:
    nama_sumber = yaml.safe_load(f)['names']
nama_sumber = [nama_sumber[i] for i in sorted(nama_sumber)] if isinstance(nama_sumber, dict) else list(nama_sumber)
idx_person = [str(n).lower() for n in nama_sumber].index('person')
for split_src, split in (('train', 'train'), ('valid', 'val'), ('test', 'val')):
    folder = K['data_person'] / split_src / 'images'
    if not folder.exists():
        continue
    n = 0
    for img in sorted(folder.iterdir()):
        if img.suffix.lower() not in EKSTENSI:
            continue
        lbl = K['data_person'] / split_src / 'labels' / f'{img.stem}.txt'
        baris = []
        if lbl.exists():
            for b in lbl.read_text().splitlines():
                q = b.split()
                if len(q) >= 5 and int(float(q[0])) == idx_person:
                    baris.append(' '.join(['0'] + q[1:5]))
        if not tulis_contoh(DATA, split, f'caviar_{img.stem}', img, baris, SISI):
            continue
        if split == 'val':   # validasi khusus kamera koridor dan lobi
            tulis_contoh(DATA, 'val_caviar', f'caviar_{img.stem}', img, baris, SISI)
        n += 1
    print(f'  dataset_yolo_person {split_src} -> {split}: {n} gambar')

# bank penghalang
buka(SUMBER['bank'], TAHAP / 'bank', KERJA)
BANK = [(p.name.split('__')[0], cv2.imread(str(p))) for p in sorted((TAHAP / 'bank').glob('*.jpg'))]
BANK = [(n, im) for n, im in BANK if im is not None]
shutil.rmtree(TAHAP / 'bank')
print('  bank penghalang:', dict(Counter(n for n, _ in BANK)))


# ---------- 5. oklusi sintetis (Simple Copy-Paste, Ghiasi dkk. CVPR 2021) ----------
def tempel(img, potongan, x1, y1, x2, y2, rng):
    w, h = x2 - x1, y2 - y1
    obj = cv2.resize(potongan, (w, h), interpolation=cv2.INTER_LINEAR)
    obj = np.clip(obj.astype(np.float32) * rng.uniform(0.8, 1.15), 0, 255)
    alfa = np.ones((h, w), np.float32)
    tepi = max(2, min(w, h) // 12)
    alfa[:tepi] *= np.linspace(0, 1, tepi)[:, None]
    alfa[-tepi:] *= np.linspace(1, 0, tepi)[:, None]
    alfa[:, :tepi] *= np.linspace(0, 1, tepi)[None]
    alfa[:, -tepi:] *= np.linspace(1, 0, tepi)[None]
    bagian = img[y1:y2, x1:x2].astype(np.float32)
    img[y1:y2, x1:x2] = (bagian * (1 - alfa[..., None]) + obj * alfa[..., None]).astype(np.uint8)


def tertutup_oleh(kotak, x1, y1, x2, y2):
    bx1, by1, bx2, by2 = kotak
    iw, ih = max(0, min(bx2, x2) - max(bx1, x1)), max(0, min(by2, y2) - max(by1, y1))
    return iw * ih / max((bx2 - bx1) * (by2 - by1), 1)


def oklusi(p, rng):
    img = cv2.imread(str(p))
    if img is None:
        return False
    lbl = baca_label(DATA / 'labels' / 'train' / f'{p.stem}.txt')
    H, W = img.shape[:2]
    orang = [l for l in lbl if l[0] == 0 and l[4] * H >= 48 and l[3] * W >= 16]
    if not orang:
        return False
    target = orang[rng.integers(len(orang))]
    c, cx, cy, w, h = target
    px1, py1, px2, py2 = int((cx - w / 2) * W), int((cy - h / 2) * H), int((cx + w / 2) * W), int((cy + h / 2) * H)
    pw, ph = px2 - px1, py2 - py1
    porsi = rng.uniform(0.40, 0.75)
    arah = rng.choice(['atas', 'bawah', 'samping'], p=[0.45, 0.35, 0.20])
    lebar_ekstra = int(pw * rng.uniform(0.2, 0.6))
    if arah == 'atas':      # hanya kaki terlihat, seperti orang di balik motor
        x1, x2, y1, y2 = px1 - lebar_ekstra, px2 + lebar_ekstra, py1 - int(ph * 0.05), py1 + int(ph * porsi)
    elif arah == 'bawah':   # hanya badan atas terlihat
        x1, x2, y1, y2 = px1 - lebar_ekstra, px2 + lebar_ekstra, py2 - int(ph * porsi), py2 + int(ph * 0.05)
    else:
        kiri = rng.random() < 0.5
        x1, x2 = (px1 - lebar_ekstra, px1 + int(pw * porsi)) if kiri else (px2 - int(pw * porsi), px2 + lebar_ekstra)
        y1, y2 = py1 + int(ph * 0.1), py2 + int(ph * 0.05)
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
    if x2 - x1 < 12 or y2 - y1 < 12:
        return False
    nama, potongan = BANK[rng.integers(len(BANK))]
    tempel(img, potongan, x1, y1, x2, y2, rng)
    baru = []
    for l in lbl:   # buang benda lain yang hampir seluruhnya tertutup tempelan; orang sasaran tetap utuh
        kotak = ((l[1] - l[3] / 2) * W, (l[2] - l[4] / 2) * H, (l[1] + l[3] / 2) * W, (l[2] + l[4] / 2) * H)
        if l is target or tertutup_oleh(kotak, x1, y1, x2, y2) < 0.8:
            baru.append(l)
    if nama in KELAS:   # motor tempelan ikut dilabeli
        baru.append([KELAS.index(nama), (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H])
    cv2.imwrite(str(p.with_name(p.stem + '_okl.jpg')), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
    (DATA / 'labels' / 'train' / f'{p.stem}_okl.txt').write_text(
        ''.join(f'{int(l[0])} {l[1]:.6f} {l[2]:.6f} {l[3]:.6f} {l[4]:.6f}\n' for l in baru))
    return True


asli = sorted(p for p in (DATA / 'images' / 'train').iterdir() if p.suffix.lower() in EKSTENSI)
rng = np.random.default_rng(SEED)
n_okl = sum(oklusi(p, rng) for p in asli if rng.random() < K['oklusi'])
print('  salinan oklusi sintetis:', n_okl)
assert n_okl >= K['min']['oklusi_latih'], 'Oklusi sintetis terlalu sedikit'


# ---------- 6. simulasi malam (inframerah) dan CCTV murah ----------
def buat_salinan(porsi, fungsi, akhiran, seed):
    rng = np.random.default_rng(seed)
    n = 0
    for p in asli:   # hanya gambar asli, bukan salinan oklusi
        if rng.random() >= porsi:
            continue
        img = cv2.imread(str(p))
        if img is None:
            continue
        cv2.imwrite(str(p.with_name(p.stem + akhiran + '.jpg')), fungsi(img, rng), [cv2.IMWRITE_JPEG_QUALITY, 92])
        shutil.copy2(DATA / 'labels' / 'train' / f'{p.stem}.txt', DATA / 'labels' / 'train' / f'{p.stem}{akhiran}.txt')
        n += 1
    return n


print('  salinan malam:', buat_salinan(K['simulasi_malam'], jadikan_malam, '_ir', SEED + 1))
print('  salinan CCTV :', buat_salinan(K['simulasi_cctv'], jadikan_cctv, '_cctv', SEED + 2))

# ---------- 7. simpan dataset jadi ke Drive ----------
print('\nMenyimpan dataset jadi ke Drive (sekali saja) ...')
kemas(DATA, FINAL, KERJA)
selesai_ringkas()

## 6. Jalankan penyiapan data

Pertama kali: sekitar 1 sampai 2 jam (unduhan). Berikutnya: beberapa menit (dari Drive). Bila sel ini error karena koneksi, jalankan lagi: sumber yang sudah selesai tidak diunduh ulang.

In [ ]:
import sys
sys.path.insert(0, '/content')
from sad_bantu import jalankan
jalankan('/content/siapkan_data.py')

## 7. Contoh hasil oklusi sintetis

Kotak biru: person. Kotak oranye: motorcycle.

In [ ]:
import sys
sys.path.insert(0, '/content')
import cv2, matplotlib.pyplot as plt
from sad_bantu import baca_label, muat_konfig
K = muat_konfig()
contoh = sorted((K['data_dir'] / 'images' / 'train').glob('*_okl.jpg'))[:4]
assert contoh, 'Belum ada data. Jalankan sel 6 dulu.'
fig, ax = plt.subplots(1, len(contoh), figsize=(4 * len(contoh), 4), squeeze=False)
for a, f in zip(ax[0], contoh):
    img = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    for l in baca_label(K['data_dir'] / 'labels' / 'train' / f'{f.stem}.txt'):
        x1, y1 = int((l[1] - l[3] / 2) * W), int((l[2] - l[4] / 2) * H)
        x2, y2 = int((l[1] + l[3] / 2) * W), int((l[2] + l[4] / 2) * H)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 90, 255) if l[0] == 0 else (255, 140, 0), 2)
    a.imshow(img)
    a.axis('off')
plt.tight_layout()
plt.show()

## 8. Skrip pelatihan

Pelatihan berjalan di proses terpisah. Checkpoint `last.pt` tersimpan di Drive setiap epoch. Skrip memeriksa status sendiri: mulai, lanjut, atau lewati.

In [ ]:
%%writefile /content/latih.py
"""Latih satu model (A atau B) di proses terpisah dari kernel notebook.

Jalankan ulang perintah yang sama kapan saja:
- belum pernah dilatih -> mulai dari yolo26s.pt
- terputus di tengah    -> lanjut dari epoch terakhir (last.pt di Drive)
- sudah selesai         -> dilewati
"""
import json, os, shutil, sys, time
from pathlib import Path

sys.path.insert(0, '/content')
from sad_bantu import DATA_VERSI, hapus_cache_label, muat_konfig, periksa_dataset, status_latih

m = sys.argv[1].upper()
assert m in ('A', 'B'), 'Pakai: python latih.py A  atau  python latih.py B'
K = muat_konfig()
yaml_info = K['kerja_dir'] / 'sad_yaml.json'
assert yaml_info.exists(), 'Data belum siap. Jalankan sel penyiapan data dulu.'
DATA_YAML = Path(json.loads(yaml_info.read_text())[m]['validasi'])
assert DATA_YAML.exists(), f'{DATA_YAML} tidak ada. Jalankan sel penyiapan data dulu.'

run = K['hasil_dir'] / K['nama'][m]
last, best = run / 'weights' / 'last.pt', run / 'weights' / 'best.pt'
status = status_latih(K, m)
print(f'Model {m} ({K["nama"][m]}): status {status}')


def arsipkan(alasan):
    cadangan = run.with_name(f"{run.name}_lama_{time.strftime('%Y%m%d_%H%M%S')}")
    shutil.move(str(run), str(cadangan))
    print(f'{alasan} Folder lama dipindah ke {cadangan.name}, pelatihan dimulai ulang.')


penanda = run / 'DATA_VERSI'
if status == 'rusak':
    arsipkan('Checkpoint tidak terbaca.')
    status = 'belum'
elif status == 'berjalan' and (not penanda.exists() or penanda.read_text().strip() != DATA_VERSI):
    arsipkan('Run lama dibuat dengan data versi lama (label motor terbuang).')
    status = 'belum'

if status != 'selesai':
    hapus_cache_label(DATA_YAML.parent)
    periksa_dataset(DATA_YAML)

from ultralytics import YOLO

if status == 'selesai':
    print(f'Model {m} sudah selesai: {best}')
elif status == 'berjalan':
    import torch
    ck = torch.load(str(last), map_location='cpu', weights_only=False)
    total = ck.get('train_args', {}).get('epochs', K['epochs'])
    print(f"{ck['epoch'] + 1} dari {total} epoch sudah selesai. Melanjutkan ke epoch {ck['epoch'] + 2}.")
    del ck
    try:
        YOLO(str(last)).train(resume=True)
    except AssertionError as e:
        if 'nothing to resume' not in str(e) or not best.exists():
            raise
        print('Semua epoch sudah dijalankan.')
else:
    run.mkdir(parents=True, exist_ok=True)
    penanda.write_text(DATA_VERSI)
    YOLO(K['model_awal']).train(
        data=str(DATA_YAML), epochs=K['epochs'], imgsz=K['imgsz'], batch=K['batch'], patience=K['patience'],
        workers=K['workers'], cache=False, project=str(K['hasil_dir']), name=K['nama'][m], exist_ok=True,
        seed=K['seed'], cos_lr=True, close_mosaic=10, hsv_v=0.5, degrees=5.0, perspective=0.0005, shear=2.0,
        scale=0.7, fliplr=0.5, mosaic=1.0, mixup=0.1, plots=True)

assert best.exists(), f'best.pt model {m} tidak ditemukan setelah pelatihan'
(run / 'SELESAI').write_text('ok')
print(f'MODEL {m} SELESAI: {best}')

## 9. Latih model A: parkiran (person + motorcycle)

Bila sesi berakhir di tengah pelatihan, **Run all** lagi. Pelatihan lanjut dari epoch terakhir.

In [ ]:
import sys
sys.path.insert(0, '/content')
from sad_bantu import jalankan
jalankan('/content/latih.py', 'A')

## 10. Latih model B: umum (person)

In [ ]:
import sys
sys.path.insert(0, '/content')
from sad_bantu import jalankan
jalankan('/content/latih.py', 'B')

## 11. Skrip evaluasi

Model baru dibandingkan dengan model lama pada data yang sama:

| Data uji | Mengukur |
| --- | --- |
| validasi | gabungan COCO, Open Images, CAVIAR |
| val_oklusi | orang tertutup sebagian (ukuran utama perbaikan) |
| val_caviar | kamera koridor dan lobi asli |
| malam, cctv (simulasi) | salinan validasi dalam kondisi malam dan CCTV murah |

**LAYAK PASANG** bila recall orang tertutup naik dan mAP50 validasi (orang, dan motor untuk model A) tidak turun lebih dari 0,02. Uji video memakai confidence yang sama dengan web (orang 0,15, motor 0,3).

In [ ]:
%%writefile /content/evaluasi.py
"""Evaluasi model lama dan baru pada data yang sama, keputusan LAYAK PASANG, dan uji video."""
import json, shutil, sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

sys.path.insert(0, '/content')
from sad_bantu import (hapus_cache_label, jadikan_cctv, jadikan_malam, model_terbaik, muat_konfig, periksa_dataset,
                       tulis_yaml, versi_person)

K = muat_konfig()
yaml_info = K['kerja_dir'] / 'sad_yaml.json'
assert yaml_info.exists(), 'Data belum siap. Jalankan sel penyiapan data dulu.'
YAMLS = json.loads(yaml_info.read_text())
BARU = {m: model_terbaik(K, m) for m in ('A', 'B')}    # berhenti dengan pesan jelas bila belum selesai
KELAS, IMGSZ, DATA, KERJA, PAKET = K['kelas'], K['imgsz'], K['data_dir'], K['kerja_dir'], K['paket_dir']
PAKET.mkdir(parents=True, exist_ok=True)
for akar in (DATA, K['data_b']):
    hapus_cache_label(akar)
for m in ('A', 'B'):
    periksa_dataset(YAMLS[m]['validasi'])

from ultralytics import YOLO


def salinan_kondisi(fungsi, nama, seed):
    """Salinan validasi dalam kondisi malam atau CCTV murah (label sama)."""
    d = KERJA / f'val_{nama}'
    if d.exists():
        shutil.rmtree(d)
    (d / 'images' / 'val').mkdir(parents=True)
    shutil.copytree(DATA / 'labels' / 'val', d / 'labels' / 'val')
    rng = np.random.default_rng(seed)
    for p in sorted((DATA / 'images' / 'val').iterdir()):
        img = cv2.imread(str(p))
        if img is not None:
            cv2.imwrite(str(d / 'images' / 'val' / (p.stem + '.jpg')), fungsi(img, rng))
    for c in (d / 'labels' / 'val').glob('*.cache'):
        c.unlink()
    db = KERJA / f'val_{nama}_person'
    versi_person(d, db)
    return {'A': str(tulis_yaml(d, 'val', KELAS, 'data.yaml')), 'B': str(tulis_yaml(db, 'val', ['person'], 'data.yaml'))}


MALAM = salinan_kondisi(jadikan_malam, 'malam', K['seed'] + 11)
CCTV = salinan_kondisi(jadikan_cctv, 'cctv', K['seed'] + 12)
UJI = {m: {**YAMLS[m], 'malam (simulasi)': MALAM[m], 'cctv (simulasi)': CCTV[m]} for m in ('A', 'B')}


def nilai(m, path_model, label):
    model = YOLO(str(path_model))
    nama = [str(v).lower() for v in model.names.values()]
    harap = KELAS if m == 'A' else ['person']
    assert nama[:len(harap)] == harap, f'Urutan kelas {path_model}: {nama}, seharusnya {harap}'
    baris = []
    for nama_data, y in UJI[m].items():
        h = model.val(data=str(y), imgsz=IMGSZ, split='val', batch=K['batch'], workers=K['workers'],
                      plots=False, verbose=False)
        for i, c in enumerate(h.box.ap_class_index):
            p, r, m50, m95 = h.box.class_result(i)
            baris.append({'model': f'{m}-{label}', 'data': nama_data, 'kelas': model.names[int(c)],
                          'precision': p, 'recall': r, 'F1': 2 * p * r / (p + r) if p + r else 0,
                          'mAP50': m50, 'mAP50-95': m95})
    return baris


baris = []
for m in ('A', 'B'):
    baris += nilai(m, K['model_lama'][m], 'lama') + nilai(m, BARU[m], 'baru')
tabel = pd.DataFrame(baris).round(4)
tabel.to_csv(PAKET / 'evaluasi_model.csv', index=False)


def ambil(model, data, kelas, kolom):
    v = tabel[(tabel.model == model) & (tabel.data == data) & (tabel.kelas == kelas)][kolom]
    return float(v.iloc[0]) if len(v) else float('nan')


keputusan = {}
for m in ('A', 'B'):
    naik_okl = ambil(f'{m}-baru', 'val_oklusi', 'person', 'recall') - ambil(f'{m}-lama', 'val_oklusi', 'person', 'recall')
    beda_val = ambil(f'{m}-baru', 'validasi', 'person', 'mAP50') - ambil(f'{m}-lama', 'validasi', 'person', 'mAP50')
    syarat = {'recall orang tertutup naik': naik_okl > 0, 'mAP50 orang validasi turun maks 0,02': beda_val >= -0.02}
    info = {'naik_recall_oklusi': naik_okl, 'beda_map50_orang': beda_val}
    if m == 'A':
        beda_motor = ambil('A-baru', 'validasi', 'motorcycle', 'mAP50') - ambil('A-lama', 'validasi', 'motorcycle', 'mAP50')
        syarat['mAP50 motor validasi turun maks 0,02'] = beda_motor >= -0.02
        info['beda_map50_motor'] = beda_motor
    keputusan[m] = {'layak': all(syarat.values()), 'syarat': syarat, **info}
    print(f"Model {m}: {'LAYAK PASANG' if keputusan[m]['layak'] else 'JANGAN PASANG'} | "
          + ', '.join(f'{k}: {"ya" if v else "TIDAK"}' for k, v in syarat.items()))
(PAKET / 'keputusan.json').write_text(json.dumps(keputusan, indent=2, ensure_ascii=False))


# ---------- uji video: persentase frame berisi orang dan motor (confidence sama dengan web) ----------
def persen_frame(path_model, path_video, per_detik=3):
    model = YOLO(str(path_model))
    nama = [str(v).lower() for v in model.names.values()]
    i_orang = nama.index('person')
    i_motor = nama.index('motorcycle') if 'motorcycle' in nama else None
    c = cv2.VideoCapture(str(path_video))
    fps = c.get(cv2.CAP_PROP_FPS) or 25
    langkah, i, n, ada_o, ada_m = max(1, round(fps / per_detik)), 0, 0, 0, 0
    while True:
        ok, f = c.read()
        if not ok:
            break
        if i % langkah == 0:
            r = model.predict(f, conf=0.15, imgsz=IMGSZ, verbose=False)[0]
            kls, cf = r.boxes.cls.int().tolist(), r.boxes.conf.tolist()
            ada_o += any(k == i_orang for k in kls)
            if i_motor is not None:
                ada_m += any(k == i_motor and s >= 0.3 for k, s in zip(kls, cf))
            n += 1
        i += 1
    c.release()
    return f'{ada_o / max(n, 1):.0%}', (f'{ada_m / max(n, 1):.0%}' if i_motor is not None else '-')


VIDEO = sorted(p for p in K['video_dir'].glob('*') if p.suffix.lower() in ('.mp4', '.avi', '.mov', '.mkv', '.mpg'))
baris = []
for v in VIDEO:
    for label, path in (('A-lama', K['model_lama']['A']), ('A-baru', BARU['A']),
                        ('B-lama', K['model_lama']['B']), ('B-baru', BARU['B'])):
        o, mtr = persen_frame(path, v)
        baris.append({'video': v.name, 'model': label, 'frame berisi orang': o, 'frame berisi motor': mtr})
    for _ in YOLO(str(BARU['A'])).predict(str(v), conf=0.15, imgsz=IMGSZ, save=True, verbose=False, stream=True,
                                          project=str(K['hasil_dir'] / K['nama']['A']), name='uji_video',
                                          exist_ok=True):
        pass
pd.DataFrame(baris).to_csv(PAKET / 'uji_video.csv', index=False)
print('EVALUASI SELESAI')

## 12. Jalankan evaluasi

In [ ]:
import sys
sys.path.insert(0, '/content')
from sad_bantu import jalankan
jalankan('/content/evaluasi.py')

## 13. Hasil evaluasi, kurva, dan confusion matrix

In [ ]:
import sys, json
sys.path.insert(0, '/content')
import pandas as pd
from IPython.display import Image, display
from sad_bantu import muat_konfig
K = muat_konfig()
assert (K['paket_dir'] / 'evaluasi_model.csv').exists(), 'Evaluasi belum ada. Jalankan sel 12 dulu.'
tabel = pd.read_csv(K['paket_dir'] / 'evaluasi_model.csv')
for m, v in json.loads((K['paket_dir'] / 'keputusan.json').read_text()).items():
    print(f"Model {m}: {'LAYAK PASANG' if v['layak'] else 'JANGAN PASANG'} | "
          + ', '.join(f'{s}: {"ya" if ok else "TIDAK"}' for s, ok in v['syarat'].items()))
display(tabel.pivot_table(index=['data', 'kelas'], columns='model', values=['recall', 'mAP50']).round(3))
display(pd.read_csv(K['paket_dir'] / 'uji_video.csv'))
for m in ('A', 'B'):
    for nama in ('results.png', 'confusion_matrix_normalized.png'):
        f = K['hasil_dir'] / K['nama'][m] / nama
        if f.exists():
            print(f'Model {m}: {nama}')
            display(Image(filename=str(f), width=850))

## 14. Paket model untuk web

Paket berisi `motor.pt`, `best.pt`, tabel evaluasi, uji video, dan `KARTU_MODEL.md`. Paket tersimpan di `MyDrive/supicious-activity-detection/model_web/` dan diunduh sebagai `model_web.zip`.

Di komputer Anda: ganti nama berkas lama di `web/models/` (misalnya `motor_lama.pt` dan `best_lama.pt`), salin berkas dari paket, lalu mulai ulang web. **Pasang hanya model berstatus LAYAK PASANG.**

In [ ]:
import sys, json, shutil
sys.path.insert(0, '/content')
from datetime import date
import pandas as pd
import ultralytics
from sad_bantu import model_terbaik, muat_konfig
K = muat_konfig()
P = K['paket_dir']
assert (P / 'keputusan.json').exists(), 'Evaluasi belum ada. Jalankan sel 12 dulu.'
keputusan = json.loads((P / 'keputusan.json').read_text())
tabel = pd.read_csv(P / 'evaluasi_model.csv')
video = pd.read_csv(P / 'uji_video.csv')
for m in ('A', 'B'):
    shutil.copy2(model_terbaik(K, m), P / K['berkas_web'][m])

def tabel_md(df):
    kolom = list(df.columns)
    baris = ['| ' + ' | '.join(kolom) + ' |', '| ' + ' | '.join('---' for _ in kolom) + ' |']
    baris += ['| ' + ' | '.join(str(v) for v in r) + ' |' for r in df.itertuples(index=False)]
    return '\n'.join(baris)

def status(m):
    return 'LAYAK PASANG' if keputusan[m]['layak'] else 'JANGAN PASANG'

n_caviar = len(list((K['data_dir'] / 'labels' / 'val_caviar').glob('*.txt')))
kartu = f"""# Kartu model web deteksi perilaku

Tanggal: {date.today().isoformat()} | Ultralytics {ultralytics.__version__} | YOLO26s, {K['imgsz']} px, {K['epochs']} epoch maks

| Berkas | Kelas | Mode web | Keputusan |
| --- | --- | --- | --- |
| motor.pt | person, motorcycle | parkiran | {status('A')} |
| best.pt | person | umum | {status('B')} |

## Data latih

| Sumber | Jumlah | Lisensi |
| --- | --- | --- |
| COCO 2017 (train2017/val2017) | latih {K['coco_latih']}, validasi {K['coco_val']} | Anotasi CC BY 4.0; gambar lisensi Flickr |
| Open Images V7 (test/validation) | latih {K['oi_latih']}, validasi {K['oi_val']} | Anotasi CC BY 4.0; gambar CC BY 2.0 |
| dataset_yolo_person_v2 (CAVIAR) | {n_caviar} gambar validasi + data latih | CC BY-SA, EC Funded CAVIAR project/IST 2001 37540 |

Augmentasi: oklusi sintetis copy-paste {K['oklusi']:.0%} (Ghiasi dkk., CVPR 2021), simulasi malam {K['simulasi_malam']:.0%}, simulasi CCTV {K['simulasi_cctv']:.0%}, ditambah augmentasi Ultralytics.

## Evaluasi

{tabel_md(tabel[['model', 'data', 'kelas', 'precision', 'recall', 'F1', 'mAP50']])}

## Uji video

{tabel_md(video)}

## Batasan

- Data parkiran dan toko dari lokasi pelanggan belum ada; kalibrasi ambang memakai penilaian petugas di web.
- Simulasi malam bukan rekaman inframerah asli.
- Lisensi Ultralytics (AGPL-3.0 atau Enterprise) berlaku untuk produk yang dijual.
"""
(P / 'KARTU_MODEL.md').write_text(kartu, encoding='utf-8')
arsip = shutil.make_archive('/content/model_web', 'zip', P)
print('Paket:', P, '| isi:', sorted(p.name for p in P.iterdir()))
from google.colab import files
files.download(arsip)